# Recurring Transaction Detection

Implements a deterministic, explainable detector for recurring transactions —
subscriptions, bills and other regular payments — built on top of
`notebooks/02_exploratory_data_analysis.ipynb`'s recurring-transaction
investigation (RQ5).

**Why this is useful.** Knowing which merchants a user pays regularly, and
how confidently, is a prerequisite for later product features — a
"Recurring" filter in Insights, a subscriptions summary, or (combined with
anomaly detection, not in this PR — see "Relationship with Anomaly
Detection" below) flagging an unexpected amount on an otherwise stable
charge.

**Critical rule this notebook follows throughout (PR-010): do not assume
repeated merchant activity is recurring.** A merchant seen more than once is
only a *candidate*. Whether it is actually recurring depends on whether the
amount and the interval between occurrences are *also* consistent — see
section 7 (Examples) for a real example (`O Pescador`) of a repeated
merchant that is correctly **not** classified as recurring.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from finance_analytics.io.csv import load_transactions_csv
from finance_analytics.recurring.detector import (
    MIN_OCCURRENCES_CANDIDATE,
    MIN_OCCURRENCES_FOR_RECURRING,
    POSSIBLE_RECURRING_CONFIDENCE_THRESHOLD,
    RECURRING_CONFIDENCE_THRESHOLD,
    detect_recurring_transactions,
)
from finance_analytics.recurring.features import build_candidate_features
from finance_analytics.recurring.scoring import (
    AMOUNT_VARIATION_TOLERANCE,
    INTERVAL_VARIATION_TOLERANCE,
    amount_consistency,
    interval_consistency,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

DATA_PATH = Path("../data/raw/finance_analytics_test_transactions.csv")

## Dataset

Same fixture and same cleaning steps as notebooks 02 and 03: drop rows with
an invalid date/amount, drop the duplicate transaction ID, and drop the QA
fixture row disguised as a transaction (`description` containing "test").
See notebook 02, section 3, for the full reasoning.

In [2]:
transactions = load_transactions_csv(DATA_PATH)

structurally_valid = transactions.dropna(subset=["date", "amount"]).drop_duplicates(subset=["id"])
synthetic_qa_mask = structurally_valid["description"].str.contains("test", case=False, na=False)
clean = structurally_valid.loc[~synthetic_qa_mask].copy()

print(f"Clean rows: {len(clean)}")
clean.head()

Clean rows: 18


,id,date,amount,currency,description,merchant,category,account
0,1,2026-02-02,-12.50,EUR,Morning coffee,Coffee Corner,Food & Dining,Main Account
1,2,2026-02-03,-54.90,EUR,Weekly groceries,Continente,Groceries,Main Account
2,3,2026-02-05,-29.99,EUR,Monthly subscription,Spotify,Subscriptions,Main Account
3,4,2026-02-07,-82.40,EUR,Dinner with friends,O Pescador,Food & Dining,Main Account
4,5,2026-02-10,-45.00,EUR,Electricity bill,EDP,Utilities,Main Account


## 1. What the EDA Revealed

From `02_exploratory_data_analysis.ipynb`, section 8 (Recurring
Transactions — RQ5), using `finance_analytics.analysis.recurring.recurring_candidates`
— the *exploratory* table that this PR's detector builds on, not replaces:

- **`Spotify` is the strongest candidate in this fixture.** Identical amount
  both times (0% amount variation) and a ~28-day gap — a pattern consistent
  with a monthly subscription.
- **`Continente` is plausible but thinner evidence.** ~7.8% amount
  variation, a ~28-day gap — a repeat grocery visit a month apart, which
  could be a habit rather than a fixed charge.
- **`O Pescador`'s amount varies by ~52%** despite a similar ~29-day gap —
  read in the EDA as casual dining out, not a fixed recurring charge, and
  the concrete example this notebook uses throughout to distinguish
  "repeated merchant" from "recurring payment".
- **A structural limitation, not just a data-size one: `interval_variation`
  is `NaN` for every candidate in this fixture.** With exactly 2
  occurrences there is exactly 1 interval — nothing to compute *variation*
  against. Seeing whether a cadence is *consistent* (not just present)
  needs at least 3 occurrences. This directly shapes section 2's minimum-history
  rules below.
- **A short observation window under-detects.** `Netflix` reads like a
  subscription from its description but appears only once in this 46-day
  window, so any approach requiring at least 2 occurrences cannot say
  anything about it yet.

This PR turns those observations into explicit, tested rules rather than
re-deriving them.

## 2. Recurring Definition

A recurring candidate combines multiple signals (PR-010, Recurring
Definition): repeated merchant, similar amount, a relatively stable
interval, and *enough* occurrences and history to trust that stability —
not merchant repetition alone.

Two explicit minimum-history rules, both justified directly by the EDA
finding above:

- **`MIN_OCCURRENCES_CANDIDATE = 2`** — fewer
  occurrences than this and there is nothing to compare against at all.
- **`MIN_OCCURRENCES_FOR_RECURRING = 3`** —
  fewer occurrences than this and interval *consistency* cannot be assessed
  (the structural `NaN` from section 1: 2 occurrences = 1 interval = no
  variation to compute). A candidate below this bar can be at best
  "Possible recurring", never "Recurring" — regardless of how tight its
  amount match is. `Spotify` in section 7 is the concrete illustration.

Frequency is described with tolerant bands, not exact-day matches (PR-010:
"do not assume 30 days == monthly exactly") — `finance_analytics.recurring.scoring.classify_frequency`
buckets `median_interval_days` into Weekly / Biweekly / Monthly / Quarterly
/ Other regular interval / Unknown. Both `Spotify` and `Continente` sit at
28 days, comfortably inside the Monthly band (26-35 days) rather than
missing it by being 2 days short of exactly 30.

In [3]:
print(f"Minimum occurrences for any candidate:   {MIN_OCCURRENCES_CANDIDATE}")
print(f"Minimum occurrences to reach 'Recurring': {MIN_OCCURRENCES_FOR_RECURRING}")

Minimum occurrences for any candidate:   2
Minimum occurrences to reach 'Recurring': 3


## 3. Candidate Generation

`finance_analytics.recurring.features.build_candidate_features` aggregates
expense transactions into one row per merchant: `occurrences`,
`total_amount`, `median_amount`, `amount_variation`, `median_interval_days`,
`interval_variation`, `first_seen`, `last_seen`, and the contributing
`transaction_ids`.

**Scope: expenses only** — a recurring transaction here means a recurring
*payment*. `Income` rows (`Employer Payroll`, `Freelance Client`) are
excluded, matching `analysis/recurring.py` and `anomalies/features.py`; the
fixture has only one salary row and one freelance-payment row, not enough
evidence to justify widening scope to income.

**No merchant-name normalisation.** The fixture's merchant names have no
case, whitespace or formatting variants — PR-010 requires only the minimum
deterministic normalisation the EDA actually shows is needed, and here
none is, so exact string match is used.

Unlike `analysis.recurring.recurring_candidates` (PR-008's exploratory
table, which drops merchants below its `min_occurrences`), every merchant
with at least one expense is kept here — including single-occurrence ones,
with `occurrences=1` and `NaN` variation stats — so section 6 can classify
them explicitly as insufficient history instead of silently omitting them.

In [4]:
candidates = build_candidate_features(clean)
candidates.drop(columns=["transaction_ids"])

,merchant,currency,occurrences,total_amount,median_amount,amount_variation,median_interval_days,interval_variation,first_seen,last_seen
0,ATM,EUR,1,100.00,100.00,NaN,NaN,NaN,2026-03-10,2026-03-10
1,CP,EUR,1,120.00,120.00,NaN,NaN,NaN,2026-02-15,2026-02-15
2,Café Central,EUR,1,8.20,8.20,NaN,NaN,NaN,2026-02-22,2026-02-22
3,Coffee Corner,EUR,1,12.50,12.50,NaN,NaN,NaN,2026-02-02,2026-02-02
4,Continente,EUR,2,116.20,58.10,0.077891,28.0,NaN,2026-02-03,2026-03-03
5,EDP,EUR,1,45.00,45.00,NaN,NaN,NaN,2026-02-10,2026-02-10
6,Farmácia Central,EUR,1,18.75,18.75,NaN,NaN,NaN,2026-02-12,2026-02-12
7,Galp,EUR,1,22.40,22.40,NaN,NaN,NaN,2026-03-12,2026-03-12
8,MediaMarkt,EUR,1,899.00,899.00,NaN,NaN,NaN,2026-02-20,2026-02-20
9,Netflix,EUR,1,15.99,15.99,NaN,NaN,NaN,2026-02-24,2026-02-24


## 4. Interval Analysis

Intervals are the day-gaps between a merchant's consecutive transactions
(chronological order). `median_interval_days` describes the typical gap;
`interval_variation` (coefficient of variation, std / mean of the gaps) says
how *consistent* that gap is — low means evenly spaced, high means erratic.

`interval_consistency` maps that coefficient to a 0-1 score, reaching 0 at
a variation of `0.4` (kept wider than the
amount tolerance below — calendar effects and billing-day drift make
timing inherently noisier than a fixed price). When `interval_variation`
is `NaN` — exactly 2 occurrences, the structural case from section 1 — the
score is a neutral midpoint (0.5): neither confirmed stable nor confirmed
unstable, not penalised or rewarded either way.

In [5]:
print(f"Interval variation tolerance (0 at/above this): {INTERVAL_VARIATION_TOLERANCE}")

interval_variation_examples = pd.DataFrame(
    {"interval_variation": [0.0, 0.1, 0.2, 0.3, 0.4, np.nan]}
)
interval_variation_examples["interval_consistency"] = interval_variation_examples[
    "interval_variation"
].apply(interval_consistency)
interval_variation_examples

Interval variation tolerance (0 at/above this): 0.4


,interval_variation,interval_consistency
0,0.0,1.00
1,0.1,0.75
2,0.2,0.50
3,0.3,0.25
4,0.4,0.00
5,NaN,0.50


## 5. Amount Analysis

Recurring payments may repeat exactly or with small variation (a price
bump, a tip). `amount_variation` is the coefficient of variation of a
merchant's transaction amounts; `amount_consistency` maps it to a 0-1
score, reaching 0 at a variation of `0.3`.

That tolerance is anchored directly to the EDA fixture: `O Pescador`'s
~52% variation must score at (or near) 0 — it reads as casual dining out,
not a fixed charge — while `Continente`'s ~7.8% should still score highly.

In [6]:
print(f"Amount variation tolerance (0 at/above this): {AMOUNT_VARIATION_TOLERANCE}")

amount_examples = candidates[["merchant", "occurrences", "amount_variation"]].dropna().copy()
amount_examples["amount_consistency"] = amount_examples["amount_variation"].apply(
    amount_consistency
)
amount_examples.sort_values("amount_consistency", ascending=False)

Amount variation tolerance (0 at/above this): 0.3


,merchant,occurrences,amount_variation,amount_consistency
11,Spotify,2,0.000000,1.000000
4,Continente,2,0.077891,0.740362
10,O Pescador,2,0.524741,0.000000


## 6. Confidence Scoring

`finance_analytics.recurring.scoring.combine_confidence` blends four
signals into a single 0-1 score: interval consistency (weight 0.35),
amount consistency (0.30), occurrence strength — how many times the
merchant has been seen (0.20) — and history-span strength — how much
elapsed time those occurrences actually cover (0.15). Interval and amount
consistency carry the most weight because they are what the EDA showed
actually distinguishes a subscription from a frequent-but-irregular
merchant; occurrence count and span corroborate *how much evidence there
is*, not *what the evidence shows*.

**This is a deterministic heuristic score, not a calibrated probability**
(PR-010: "Do not call it a calibrated probability unless it actually is").
Nothing here has been fit or validated against labelled outcomes — it is a
relative ranking of evidence strength, not a statement like "73% likely to
be a subscription".

Classification then applies explicit, documented thresholds to that score:

```text
occurrences < 2                                    -> Insufficient history
occurrences >= 3 and confidence >= 0.7                  -> Recurring
confidence >= 0.45 (but not the above)              -> Possible recurring
otherwise                                      -> Not recurring
```

In [7]:
print(f"Recurring confidence threshold:          >= {RECURRING_CONFIDENCE_THRESHOLD}")
print(f"Possible-recurring confidence threshold:  >= {POSSIBLE_RECURRING_CONFIDENCE_THRESHOLD}")

results = detect_recurring_transactions(clean)
results_frame = pd.DataFrame([r.__dict__ for r in results]).drop(columns=["transaction_ids"])
results_frame.sort_values(["classification", "confidence_score"], ascending=[True, False])

Recurring confidence threshold:          >= 0.7
Possible-recurring confidence threshold:  >= 0.45


,merchant,is_recurring,classification,confidence_score,frequency,occurrences,median_amount,amount_variation,median_interval_days,interval_variation,first_seen,last_seen,reason
0,ATM,False,Insufficient history,NaN,Unknown,1,100.00,NaN,NaN,None,2026-03-10,2026-03-10,ATM has only 1 transaction — at least 2 are ne...
1,CP,False,Insufficient history,NaN,Unknown,1,120.00,NaN,NaN,None,2026-02-15,2026-02-15,CP has only 1 transaction — at least 2 are nee...
2,Café Central,False,Insufficient history,NaN,Unknown,1,8.20,NaN,NaN,None,2026-02-22,2026-02-22,Café Central has only 1 transaction — at least...
3,Coffee Corner,False,Insufficient history,NaN,Unknown,1,12.50,NaN,NaN,None,2026-02-02,2026-02-02,Coffee Corner has only 1 transaction — at leas...
5,EDP,False,Insufficient history,NaN,Unknown,1,45.00,NaN,NaN,None,2026-02-10,2026-02-10,EDP has only 1 transaction — at least 2 are ne...
6,Farmácia Central,False,Insufficient history,NaN,Unknown,1,18.75,NaN,NaN,None,2026-02-12,2026-02-12,Farmácia Central has only 1 transaction — at l...
7,Galp,False,Insufficient history,NaN,Unknown,1,22.40,NaN,NaN,None,2026-03-12,2026-03-12,Galp has only 1 transaction — at least 2 are n...
8,MediaMarkt,False,Insufficient history,NaN,Unknown,1,899.00,NaN,NaN,None,2026-02-20,2026-02-20,MediaMarkt has only 1 transaction — at least 2...
9,Netflix,False,Insufficient history,NaN,Unknown,1,15.99,NaN,NaN,None,2026-02-24,2026-02-24,Netflix has only 1 transaction — at least 2 ar...
12,TAP Air,False,Insufficient history,NaN,Unknown,1,642.50,NaN,NaN,None,2026-02-18,2026-02-18,TAP Air has only 1 transaction — at least 2 ar...


In [8]:
chart_frame = results_frame.dropna(subset=["confidence_score"]).sort_values("confidence_score")
fig = px.bar(
    chart_frame,
    x="confidence_score",
    y="merchant",
    color="classification",
    orientation="h",
    title="Recurring confidence score by merchant (candidates with >=2 occurrences)",
    color_discrete_map={
        "Recurring": "#3F7D52",
        "Possible recurring": "#8A5A06",
        "Not recurring": "#AE1800",
    },
)
fig.add_vline(x=0.7, line_dash="dot", annotation_text="Recurring threshold")
fig.add_vline(x=0.45, line_dash="dot", annotation_text="Possible recurring threshold")
fig.update_layout(height=320)
fig.show()

## 7. Examples

### From the real fixture

`Spotify` and `Continente` are this dataset's real repeated merchants.
Neither reaches "Recurring" — both have only 2 occurrences, so interval
consistency is structurally unconfirmed (section 2) — but both are
correctly distinguished from `O Pescador`, whose repeat is not amount-stable.

In [9]:
for merchant in ["Spotify", "Continente", "O Pescador"]:
    result = next(r for r in results if r.merchant == merchant)
    print(f"{result.merchant:12s} {result.classification:20s} confidence={result.confidence_score}")
    print(f"  {result.reason}")

Spotify      Possible recurring   confidence=0.625
  Spotify appears 2 times, about every ~28 days with a similar amount around 29.99 EUR, but at least 3 occurrences are needed to confirm a stable interval.
Continente   Possible recurring   confidence=0.5471
  Continente appears 2 times, about every ~28 days with a similar amount around 58.10 EUR, but at least 3 occurrences are needed to confirm a stable interval.
O Pescador   Not recurring        confidence=0.325
  O Pescador appears 2 times, but timing and amounts are too variable to classify as recurring (amount varies by 52%).


### Controlled synthetic examples

No ground-truth recurring-payment labels exist for this dataset (or for
real user data). Per PR-010's Evaluation section: use controlled synthetic
cases instead of claiming accuracy. Three scenarios, matching
`tests/test_recurring_detector.py`:

**A — a clear recurring subscription** (identical amount, exact monthly
cadence, enough occurrences to confirm interval consistency).

In [10]:
synthetic_subscription = pd.DataFrame(
    [
        {
            "id": f"syn-{i}",
            "date": date,
            "amount": -12.99,
            "merchant": "StreamPlus",
            "currency": "EUR",
        }
        for i, date in enumerate(pd.date_range("2026-01-01", periods=5, freq="30D"))
    ]
)
scenario_a = next(
    r for r in detect_recurring_transactions(synthetic_subscription) if r.merchant == "StreamPlus"
)
scenario_a

RecurringResult(merchant='StreamPlus', is_recurring=True, classification='Recurring', confidence_score=1.0, frequency='Monthly', occurrences=5, median_amount=12.99, amount_variation=0.0, median_interval_days=30.0, interval_variation=0.0, first_seen=Timestamp('2026-01-01 00:00:00'), last_seen=Timestamp('2026-05-01 00:00:00'), reason='StreamPlus appears 5 times with a stable monthly interval (~30 days) and a consistent amount of 12.99 EUR.', transaction_ids=['syn-0', 'syn-1', 'syn-2', 'syn-3', 'syn-4'])

**B — a frequent but irregular merchant** (6 visits, but wildly different
amounts and gaps — repeated merchant activity that PR-010's Critical Rule
explicitly warns must not be assumed recurring).

In [11]:
irregular_dates = [
    "2026-01-01",
    "2026-01-06",
    "2026-01-19",
    "2026-02-20",
    "2026-03-10",
    "2026-03-25",
]
irregular_amounts = [-10.0, -25.0, -8.0, -42.0, -15.0, -30.0]
synthetic_irregular = pd.DataFrame(
    [
        {
            "id": f"irr-{i}",
            "date": pd.Timestamp(date),
            "amount": amount,
            "merchant": "Corner Bistro",
            "currency": "EUR",
        }
        for i, (date, amount) in enumerate(zip(irregular_dates, irregular_amounts))
    ]
)
scenario_b = next(
    r for r in detect_recurring_transactions(synthetic_irregular) if r.merchant == "Corner Bistro"
)
scenario_b

RecurringResult(merchant='Corner Bistro', is_recurring=False, classification='Not recurring', confidence_score=0.35, frequency='Biweekly', occurrences=6, median_amount=20.0, amount_variation=0.6058, median_interval_days=15.0, interval_variation=0.5942, first_seen=Timestamp('2026-01-01 00:00:00'), last_seen=Timestamp('2026-03-25 00:00:00'), reason='Corner Bistro appears 6 times, but timing and amounts are too variable to classify as recurring (amount varies by 61%, interval varies by 59%).', transaction_ids=['irr-0', 'irr-1', 'irr-2', 'irr-3', 'irr-4', 'irr-5'])

## 8. False-Positive Inspection

**Within the real fixture:** nothing reaches "Recurring" at all (the
18-row, 46-day fixture has no merchant with 3+ occurrences), so there is no
false "Recurring" verdict to inspect here. `O Pescador` is the relevant
case — a genuinely repeated merchant, correctly kept out of "Possible
recurring" territory (confidence 0.325, below the 0.45 bar) by its amount
instability alone.

**A harder, synthetic stress test:** two transactions to the same merchant,
far apart, with a *coincidentally* identical amount and no real cadence.

In [12]:
synthetic_coincidence = pd.DataFrame(
    [
        {
            "id": "b1",
            "date": pd.Timestamp("2026-01-03"),
            "amount": -5.00,
            "merchant": "Bakery",
            "currency": "EUR",
        },
        {
            "id": "b2",
            "date": pd.Timestamp("2026-04-08"),
            "amount": -5.00,
            "merchant": "Bakery",
            "currency": "EUR",
        },
    ]
)
scenario_c = next(
    r for r in detect_recurring_transactions(synthetic_coincidence) if r.merchant == "Bakery"
)
scenario_c

RecurringResult(merchant='Bakery', is_recurring=False, classification='Possible recurring', confidence_score=0.625, frequency='Quarterly', occurrences=2, median_amount=5.0, amount_variation=0.0, median_interval_days=95.0, interval_variation=None, first_seen=Timestamp('2026-01-03 00:00:00'), last_seen=Timestamp('2026-04-08 00:00:00'), reason='Bakery appears 2 times, about every ~95 days with a similar amount around 5.00 EUR, but at least 3 occurrences are needed to confirm a stable interval.', transaction_ids=['b1', 'b2'])

This is classified `Possible recurring` (confidence ~0.63) purely because
the amount happens to match — a real, honest limitation: at exactly 2
occurrences, a coincidentally identical amount is enough to clear the
"Possible recurring" bar even with a 95-day gap and no real cadence behind
it (`classify_frequency` even labels the gap "Quarterly", since 95 days
falls inside that band by coincidence too). It can never reach
"Recurring" — the `MIN_OCCURRENCES_FOR_RECURRING = 3`
floor is structural, not a confidence question — and the `reason` text is
explicit that a third occurrence is needed to confirm anything. But a
consumer reading only `is_recurring`/`classification`/`frequency` without
the `reason` sentence could still overtrust a coincidence. This is
recorded as a known limitation in section 9, not silently accepted.

## 9. Limitations

- **Dataset size and span.** 18 clean transactions over 46 days. No
  merchant in the real fixture has 3+ occurrences, so "Recurring" is never
  reached on real data in this notebook — only demonstrated synthetically
  (section 7). The thresholds are chosen to be defensible on general
  principles and consistent with the EDA's own qualitative read, not tuned
  against a large labelled dataset, because none exists.
- **No ground-truth recurring labels.** As with PR-009's anomaly detector,
  there is no labelled "this is actually a subscription" dataset to compute
  precision/recall against — evaluation here is controlled synthetic cases
  and manual inspection, not a claimed accuracy figure.
- **2-occurrence candidates carry a structural blind spot.** As section 8
  shows, a coincidentally identical amount at 2 far-apart occurrences can
  reach "Possible recurring" — capped below "Recurring", but still a
  softer signal than the label alone suggests without reading `reason`.
- **A short observation window under-detects known subscriptions.**
  `Netflix` (1 occurrence in this window) cannot be evaluated at all under
  `MIN_OCCURRENCES_CANDIDATE`. This is a property of the data span, not a
  detector flaw — a longer export would very likely surface it.
- **No merchant-name normalisation.** If a real export has formatting
  variants of the same merchant (e.g. `"AMZN*Prime"` vs `"Amazon Prime"`),
  they would be treated as different merchants and neither would
  accumulate enough history. Out of scope for this PR (see "Out of Scope"
  below); revisit only if a future dataset's EDA shows it is needed.
- **Confidence weights and tolerances are a considered heuristic, not a
  fitted model.** They are internally consistent and match the EDA's own
  qualitative judgements on this fixture (section 1), but have not been
  validated against independent data.

## 10. Next Steps

- **More history.** A longer transaction export would let the
  `MIN_OCCURRENCES_FOR_RECURRING` bar actually be exercised on real data,
  and would surface merchants like `Netflix` that this window is too short
  to say anything about.
- **Combine with anomaly detection.** PR-010's "Relationship with Anomaly
  Detection" section sketches a future insight: a recurring payment whose
  latest amount is itself anomalous (PR-009). Not implemented here — the
  two modules are deliberately independent (`recurring/` does not import
  `anomalies/`, and vice versa).
- **Revisit merchant normalisation if a future dataset needs it** — only
  with EDA evidence of actual formatting variants, per PR-010's Merchant
  Normalisation rule.
- **Surface results to an Insights layer** (Android display, a
  "Subscriptions" view) — explicitly out of scope for this PR; the
  `transaction_ids` field on each `RecurringResult` exists specifically so
  a future consumer can link a classification back to the transactions
  that produced it without re-deriving the grouping.

## Out of Scope — Confirmation

Not implemented in this notebook or in `finance_analytics.recurring`: a
full merchant-normalisation system, ML or LLM classification,
recommendations, financial advice, Room/database persistence, an API, an
Android UI, forecasting, and the combined recurring+anomaly insight
sketched in section 10. Every `reason` string above is a fixed template
(`recurring/explanations.py`) — no LLM is used anywhere in this module.